In [93]:
import requests
import pandas as pd
import numpy as np

Прочитайте JSON-файл, сохраненный в ex02.

- Один из столбцов — это число с плавающей точкой, поэтому давайте определим его формат в Pandas с помощью pd.options.display.float_format: числа с плавающей точкой должны отображаться с двумя десятичными знаками.
- В модели отсутствуют некоторые значения; ничего с ними не делайте.

In [94]:
df = pd.read_json('../ex02/auto.json')
pd.options.display.float_format = '{:.2f}'.format

In [95]:
df.head()

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus


Обогатите фрейм данных, используя выборку из этого фрейма данных.

1. Создайте выборку из 200 новых наблюдений, используя random_state = 21.
- Выборка не должна содержать новых комбинаций car number, make, и model, поэтому весь набор данных будет согласован в этом отношении.
- refund Для столбцов «и» ограничений нет fines. Вы можете выбрать случайное значение из этих столбцов и применить его к любому номеру автомобиля.
2. Объедините выборку с исходным фреймом данных, чтобы создать новый фрейм данных concat_rows.

In [96]:
df_sample = df.sample(n = 200, random_state=21)

In [97]:
concat_rows = pd.concat([df, df_sample], ignore_index=True)

concat_rows.count()

CarNumber    925
Refund       925
Fines        925
Make         925
Model        914
dtype: int64

Дополните concat_rows фрейм данных новым столбцом, содержащим сгенерированные данные.

- Создайте ряд под названием «Год» со случайными целыми числами от 1980 до 2019.
- Использовать np.random.seed(21)перед генерацией лет.
- Объедините ряд с фреймом данных и назовите его fines.

In [98]:
np.random.seed(21)
year = np.random.randint(1980, 2020, size = len(concat_rows))
concat_rows['Year'] = year

In [99]:
concat_rows.head()

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995
2,7184TT36RUS,1,2100.00,Ford,Focus,1984
3,X582HE161RUS,2,2000.00,Ford,Focus,2015
4,92918M178RUS,1,5700.00,Ford,Focus,2014


Создайте новый фрейм данных с номерами автомобилей и их владельцами.
- Получите самые популярные фамилии в США (файл surname.json можно найти в папке datasets).
- Создайте новую серию с фамилиями. Они не должны содержать спецсимволов, таких как запятые или скобки. Количество должно быть равно количеству уникальных автомобильных номеров в выборке (используйте random_state = 21).
- Создайте фрейм данных owners с двумя столбцами: CarNumber и SURNAME.

In [100]:
fines = concat_rows.copy()

In [101]:
surnames = pd.read_json('../../datasets/surname.json')

In [102]:
surnames.columns = surnames.iloc[0]

In [103]:
surnames = surnames.drop(0).reset_index(drop=True)

In [104]:
surnames = surnames['NAME'].replace('(),', '')

In [105]:
uniq_cars = fines['CarNumber'].unique()

In [106]:
np.random.seed(21)

chosen_surnames = surnames.sample(n=len(uniq_cars), random_state=21, replace=True).reset_index(drop=True)

In [107]:
chosen_surnames.head()

0    RICHARDSON
1          ROSS
2        MORGAN
3        BAILEY
4         LOPEZ
Name: NAME, dtype: object

In [108]:
owners = pd.DataFrame({
    'CarNumber':uniq_cars, 
    'SURNAME':chosen_surnames})

- Append five more observations to the fines dataframe. Come up with your own ideas for CarNumber, etc.
- Delete the last 20 observations from the owners dataframe and add three new observations that are not the same as those added to the fines dataframe.

In [110]:
extra_fines = pd.DataFrame([
    {"CarNumber": "NEWCAR001", "Refund": 1.0, "Fines": 5000.0, "Make": "Tesla", "Model": "Model3", "Year": 2018},
    {"CarNumber": "NEWCAR002", "Refund": 0.0, "Fines": 2500.0, "Make": "Toyota", "Model": "Prius", "Year": 2015},
    {"CarNumber": "NEWCAR003", "Refund": 2.0, "Fines": 1200.0, "Make": "BMW", "Model": "320i", "Year": 2010},
    {"CarNumber": "NEWCAR004", "Refund": 1.0, "Fines": 3200.0, "Make": "Ford", "Model": "Fiesta", "Year": 2008},
    {"CarNumber": "NEWCAR005", "Refund": 0.0, "Fines": 4300.0, "Make": "Audi", "Model": "A4", "Year": 2012},
])
fines = pd.concat([fines, extra_fines], ignore_index=True)

In [111]:
owners = owners.iloc[:-20, :]
extra_owners = pd.DataFrame([
    {"CarNumber": "OWNCAR001", "SURNAME": "Smith"},
    {"CarNumber": "OWNCAR002", "SURNAME": "Johnson"},
    {"CarNumber": "OWNCAR003", "SURNAME": "Williams"},
])
owners = pd.concat([owners, extra_owners], ignore_index=True)

Объедините два фрейма данных.
- Новый фрейм данных должен содержать только те номера автомобилей, которые существуют в обоих фреймах данных.
- Новый фрейм данных должен содержать все номера автомобилей из обоих фреймов данных.
- Новый фрейм данных должен содержать только номера автомобилей из fines фрейма данных.
- Новый фрейм данных должен содержать только номера автомобилей из owner sфрейма данных.

In [112]:
inner = pd.merge(fines, owners, on="CarNumber", how="inner")
outer = pd.merge(fines, owners, on="CarNumber", how="outer")
left = pd.merge(fines, owners, on="CarNumber", how="left")
right = pd.merge(fines, owners, on="CarNumber", how="right")

Создайте сводную таблицу на fines основе данных из фрейма. Она должна выглядеть примерно так (значения — это суммы штрафов), но с учётом всех лет. У вас значения могут быть другими.

In [116]:
pd.pivot_table(fines, index = ['Make', 'Model'], columns='Year', values='Fines')


Year                   1980     1981    1982    1983     1984     1985  \
Make       Model                                                         
Audi       A4           NaN      NaN     NaN     NaN      NaN      NaN   
BMW        320i         NaN      NaN     NaN     NaN      NaN      NaN   
Ford       Fiesta       NaN      NaN     NaN     NaN      NaN      NaN   
           Focus    6487.92 18590.17 8730.72 4984.62  6465.94  7394.72   
           Mondeo       NaN      NaN     NaN     NaN      NaN      NaN   
Skoda      Octavia  1200.00      NaN 2433.33 3864.86      NaN  3431.53   
Tesla      Model3       NaN      NaN     NaN     NaN      NaN      NaN   
Toyota     Camry   12000.00  8594.59     NaN 7200.00      NaN      NaN   
           Corolla      NaN      NaN 2000.00     NaN      NaN      NaN   
           Prius        NaN      NaN     NaN     NaN      NaN      NaN   
Volkswagen Golf    30900.00      NaN     NaN 8594.59   300.00 24000.00   
           Jetta        NaN      NaN     NaN     NaN      NaN      NaN   
           Passat       NaN  1600.00     NaN 3200.00 10000.00  5000.00   
           Touareg      NaN      NaN     NaN     NaN      NaN  5800.00   

Year                   1986     1987    1988     1989  ...     2010    2011  \
Make       Model                                       ...                    
Audi       A4           NaN      NaN     NaN      NaN  ...      NaN     NaN   
BMW        320i         NaN      NaN     NaN      NaN  ...  1200.00     NaN   
Ford       Fiesta       NaN      NaN     NaN      NaN  ...      NaN     NaN   
           Focus    6899.23  8380.00 6210.51  8804.73  ...  7926.57 5446.23   
           Mondeo       NaN      NaN     NaN  8600.00  ...      NaN     NaN   
Skoda      Octavia   600.00  5200.00 5200.00 45700.00  ...  1550.00  500.00   
Tesla      Model3       NaN      NaN     NaN      NaN  ...      NaN     NaN   
Toyota     Camry        NaN      NaN     NaN 22400.00  ...      NaN     NaN   
           Corolla      NaN  7450.00     NaN  4000.00  ... 24000.00 8594.59   
           Prius        NaN      NaN     NaN      NaN  ...      NaN     NaN   
Volkswagen Golf         NaN 14933.33     NaN  5800.00  ...      NaN  300.00   
           Jetta        NaN      NaN     NaN      NaN  ...      NaN     NaN   
           Passat  15000.00  6150.00     NaN      NaN  ...  1400.00     NaN   
           Touareg      NaN      NaN     NaN      NaN  ...  6300.00     NaN   

Year                   2012     2013    2014     2015     2016     2017  \
Make       Model                                                          
Audi       A4       4300.00      NaN     NaN      NaN      NaN      NaN   
BMW        320i         NaN      NaN     NaN      NaN      NaN      NaN   
Ford       Fiesta       NaN      NaN     NaN      NaN      NaN      NaN   
           Focus    5711.76  4806.71 6456.76  9957.14  6130.57 11434.78   
           Mondeo  34400.00      NaN     NaN      NaN 46200.00      NaN   
Skoda      Octavia   500.00  4198.20  300.00 23197.29   300.00  8594.59   
Tesla      Model3       NaN      NaN     NaN      NaN      NaN      NaN   
Toyota     Camry    8594.59      NaN 1000.00      NaN      NaN      NaN   
           Corolla 30300.00      NaN     NaN      NaN   900.00  9600.00   
           Prius        NaN      NaN     NaN  2500.00      NaN      NaN   
Volkswagen Golf         NaN 20800.00     NaN  2300.00      NaN      NaN   
           Jetta        NaN      NaN     NaN      NaN      NaN      NaN   
           Passat       NaN      NaN     NaN   600.00  1050.00      NaN   
           Touareg      NaN      NaN 1300.00   500.00      NaN      NaN   

Year                   2018    2019  
Make       Model                     
Audi       A4           NaN     NaN  
BMW        320i         NaN     NaN  
Ford       Fiesta       NaN     NaN  
           Focus   21083.78 4930.57  
           Mondeo       NaN     NaN  
Skoda      Octavia 52066.67 3166.67  
Tesla      Model3   5000.00     NaN  
Toyota 

Сохраните оба кадра fines данных owners в CSV-файлы без индекса.

In [117]:
fines.to_csv("fines.csv", index=False)
owners.to_csv("owners.csv", index=False)